In [ ]:
# 1. Prerequisites

In [1]:
from utils.data import load_data, save_data
from utils.plotting import create_post_stim_raster_plot, map_colour_to_electrode

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
import getpass# Load the specific network here

In [4]:
# Load the specific network here
Network = 5
DIV     = 21
group_data  = True
test_data = False

In [ ]:
data = load_data(Network, DIV, group_data)
stimulation_parameters, stimulation_patterns, binned_spike_train_responses, stimulation_times, impedance_map, electrodes = data
print(f"Loaded data with response shape {binned_spike_train_responses.shape} and parameter shape {stimulation_parameters.shape}.")
if test_data:
    data = load_data(Network, DIV, group_data, test_mode=True)
    stimulation_parameters_test, stimulation_patterns_test, binned_spike_train_responses_test, stimulation_times_test, _, _ = data
else:
    binned_spike_train_responses_test = stimulation_patterns_test = stimulation_parameters_test = stimulation_times_test = None

In [ ]:
# 2. Model Definitions and Data Preparation

In [ ]:
class CNN(nn.Module):
    """
    A simple CNN that takes an inputs of size (Batch, Channel, Height, Width). 
    The height in our task corresponds to the number of channels, the width to the number of time bins.
    """

    def __init__(self, n_outputs=1, n_base_channels= 32):
        super().__init__()

        self.convolutional_layers = nn.Sequential(
            nn.Conv2d(1, n_base_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(n_base_channels, n_base_channels*2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(n_base_channels*2, n_base_channels*4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # This function pools an arbitrary input size to a predefined size
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        # output layers
        self.fully_connected_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_base_channels * 4 * 4 * 4, n_base_channels*8),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(n_base_channels*8, n_outputs),
        )

    def forward(self, x):
        x = self.convolutional_layers(x)
        x = self.adaptive_pool(x) 
        x = self.fully_connected_layers(x)
        return x